# 1. Objective and Scope

## 1.1 Objective

The objective of this notebook is to transform the validated document page records produced by the existing ingestion pipeline into smaller, coherent, traceable text chunks suitable for subsequent embedding and retrieval.

The implementation will explore and compare two chunking strategies:

* **Fixed-Window Chunking:** Divides page content into consecutive character-based windows with configurable overlap.
* **Structure-Aware Recursive Chunking:** Splits content using meaningful document boundaries, prioritizing Markdown sections, paragraphs, lines, and sentences.

## 1.2 Existing Infrastructure

This project already provides a document ingestion layer responsible for:

* Document discovery and text extraction.
* Document and page schema validation.
* Document identification and content hashing.
* Duplicate and unchanged-document detection.
* JSONL persistence.

These capabilities will be reused without reimplementation.

The chunking process will consume the existing:

* `DocumentRecord` records from `documents.jsonl`.
* `PageRecord` records from `pages.jsonl`.

Raw source files will not be rediscovered or re-extracted.

## 1.3 Scope

This notebook covers:

1. Loading and auditing existing processed records.
2. Defining the chunk data contract and configuration.
3. Implementing fixed-window chunking.
4. Parsing Markdown document structure.
5. Implementing structure-aware recursive chunking.
6. Preserving source provenance and character offsets.
7. Generating deterministic chunk identifiers.
8. Comparing chunking strategies.
9. Persisting and validating chunk records.
10. Evaluating implementation correctness through automated tests.

## 1.4 Initial Configuration

The initial experimental configuration is:

| Parameter         |            Initial Value |
| ----------------- | -----------------------: |
| Target Chunk Size |         1,200 characters |
| Target Overlap    |           180 characters |
| Chunk Length Unit | Python string characters |
| Input Unit        |    Individual PageRecord |
| Output Format     |                    JSONL |

These values are provisional experimental settings. They are not assumed to be optimal and will require validation against the selected embedding model's tokenizer.

## 1.5 Scope Exclusions

The following capabilities are outside the scope of this notebook:

* Document discovery and extraction.
* PDF parsing.
* Embedding generation.
* Vector database construction.
* Similarity search and retrieval.
* Reranking and query rewriting.
* LLM-based question answering.

## 1.6 Expected Output

The primary output is:

`data/knowledge_base/processed/chunks.jsonl`

Every generated chunk must retain its document identity, page number, source character offsets, content hash, chunking strategy, and configuration metadata.

The final objective is to produce reproducible, validated, source-traceable chunks ready for the embedding stage.


# 2. Load Existing Processed Records

## 2.1 Objective

Load the previously ingested document and page records from the existing JSONL storage.

This step reuses the ingestion module's existing loading functions and Pydantic models. No document discovery, extraction, or ingestion operations are performed.

## 2.2 Input Files

* `data/knowledge_base/processed/documents.jsonl`
* `data/knowledge_base/processed/pages.jsonl`

## 2.3 Loading Strategy

The existing functions are responsible for reading JSONL records and validating them against their corresponding Pydantic models:

* `load_existing_documents()`
* `load_existing_pages()`

The resulting objects will be used directly by the chunking experiments.

## 2.4 Expected Data Structures

* Documents: A dictionary indexed by `document_id`.
* Pages: A list of `PageRecord` objects.

## 2.5 Validation Objective

Confirm that both datasets are successfully loaded and that the returned objects use the expected model types.


In [2]:
import sys
from pathlib import Path

# Locate the project root from the current working directory.
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = next(
    (
        path
        for path in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (path / "app" / "ingestion").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Expected an 'app/ingestion' directory."
    )

# Add the project root to Python's module search path.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import existing ingestion models and loaders.
from app.ingestion.models import DocumentRecord, PageRecord
from app.ingestion.pipeline import (
    load_existing_documents,
    load_existing_pages,
)

# Define processed data paths.
PROCESSED_DIR = PROJECT_ROOT / "data" / "knowledge_base" / "processed"

DOCUMENTS_PATH = PROCESSED_DIR / "documents.jsonl"
PAGES_PATH = PROCESSED_DIR / "pages.jsonl"

# Load existing records.
documents = load_existing_documents(DOCUMENTS_PATH)
pages = load_existing_pages(PAGES_PATH)

# Display loading results.
print("PROJECT CONFIGURATION")
print("=" * 50)
print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {CURRENT_DIR}")

print("\nDOCUMENT LOADING")
print("=" * 50)
print(f"Documents loaded: {len(documents)}")
print(f"Pages loaded: {len(pages)}")

print("\nMODEL VALIDATION")
print("=" * 50)
print(
    "All documents are DocumentRecord:",
    all(isinstance(record, DocumentRecord) for record in documents.values()),
)
print(
    "All pages are PageRecord:",
    all(isinstance(record, PageRecord) for record in pages),
)

print("\nINPUT FILES")
print("=" * 50)
print(f"Documents: {DOCUMENTS_PATH}")
print(f"Pages: {PAGES_PATH}")

PROJECT CONFIGURATION
Project root: d:\Projects\Local Mini RAG Document QA System
Working directory: d:\Projects\Local Mini RAG Document QA System\notebooks

DOCUMENT LOADING
Documents loaded: 6
Pages loaded: 6

MODEL VALIDATION
All documents are DocumentRecord: True
All pages are PageRecord: True

INPUT FILES
Documents: d:\Projects\Local Mini RAG Document QA System\data\knowledge_base\processed\documents.jsonl
Pages: d:\Projects\Local Mini RAG Document QA System\data\knowledge_base\processed\pages.jsonl


### Result Interpretation — Loading Existing Records

The processed ingestion records were loaded successfully.

| Metric                    | Result |
| ------------------------- | -----: |
| Documents loaded          |      6 |
| Pages loaded              |      6 |
| DocumentRecord validation | Passed |
| PageRecord validation     | Passed |

The notebook successfully located the project root and imported the existing ingestion modules.

**Finding:** The chunking stage can reuse the existing processed records directly. Repeating document discovery or text extraction is unnecessary.

**Next Step:** Validate the integrity, relationships, and text characteristics of the loaded corpus before implementing chunking strategies.


# 3. Input Validation and Corpus Audit

## Objective

Validate the integrity and consistency of the previously processed document and page records.

This audit examines:

1. Document identifiers and content integrity.
2. Page identifiers and document references.
3. Empty or invalid text records.
4. Page numbering and document relationships.
5. Corpus-level text statistics.

## Validation Principles

* Every document must have a unique identifier.
* Every page must reference an existing document.
* Page content must be a valid string.
* Empty pages must be identified.
* Page numbering must be inspected before defining chunk boundaries.
* All statistics must be calculated from the actual processed corpus.

No document extraction or text transformation will be performed during this stage.

In [ ]:
from collections import Counter
from statistics import mean

# --------------------------------------------------
# 1. Document Integrity
# --------------------------------------------------

document_ids = [document.document_id for document in documents.values()]

document_id_counts = Counter(document_ids)

duplicate_document_ids = [
    document_id for document_id, count in document_id_counts.items() if count > 1
]

empty_document_content = [
    document.document_id
    for document in documents.values()
    if not document.content.strip()
]

# --------------------------------------------------
# 2. Page Integrity
# --------------------------------------------------

page_document_ids = [page.document_id for page in pages]

page_id_counts = Counter((page.document_id, page.page_number) for page in pages)

duplicate_pages = [page_id for page_id, count in page_id_counts.items() if count > 1]

missing_document_references = [
    page.document_id for page in pages if page.document_id not in documents
]

empty_pages = [
    (page.document_id, page.page_number) for page in pages if not page.content.strip()
]

# --------------------------------------------------
# 3. Page Numbering
# --------------------------------------------------

pages_by_document = {}

for page in pages:
    pages_by_document.setdefault(page.document_id, []).append(page.page_number)

page_numbering = {}

for document_id, page_numbers in pages_by_document.items():
    sorted_numbers = sorted(page_numbers)

    page_numbering[document_id] = {
        "page_numbers": sorted_numbers,
        "is_sorted_contiguous": (
            sorted_numbers
            == list(
                range(
                    sorted_numbers[0],
                    sorted_numbers[0] + len(sorted_numbers),
                )
            )
        ),
    }

# --------------------------------------------------
# 4. Text Statistics
# --------------------------------------------------

page_lengths = [len(page.content) for page in pages]

document_lengths = [len(document.content) for document in documents.values()]

total_page_characters = sum(page_lengths)
total_document_characters = sum(document_lengths)

# --------------------------------------------------
# 5. Document-to-Page Relationships
# --------------------------------------------------

pages_per_document = Counter(page.document_id for page in pages)

documents_without_pages = [
    document_id for document_id in documents if pages_per_document[document_id] == 0
]

# --------------------------------------------------
# 6. Audit Report
# --------------------------------------------------

print("DOCUMENT INTEGRITY")
print("=" * 55)
print(f"Unique document IDs: {len(set(document_ids))}")
print(f"Duplicate document IDs: {duplicate_document_ids}")
print(f"Documents with empty content: {empty_document_content}")

print("\nPAGE INTEGRITY")
print("=" * 55)
print(f"Total pages: {len(pages)}")
print(f"Duplicate pages: {duplicate_pages}")
print(f"Missing document references: {missing_document_references}")
print(f"Empty pages: {empty_pages}")

print("\nPAGE NUMBERING")
print("=" * 55)

for document_id, result in page_numbering.items():
    print(f"{document_id}: {result}")

print("\nDOCUMENT-PAGE RELATIONSHIPS")
print("=" * 55)
print(f"Documents without pages: {documents_without_pages}")

for document_id, count in pages_per_document.items():
    print(f"{document_id}: {count} page(s)")

print("\nCORPUS STATISTICS")
print("=" * 55)
print(f"Total document characters: {total_document_characters:,}")
print(f"Total page characters: {total_page_characters:,}")
print(f"Minimum page length: {min(page_lengths):,}")
print(f"Maximum page length: {max(page_lengths):,}")
print(f"Average page length: {mean(page_lengths):,.2f}")
print(f"Minimum document length: {min(document_lengths):,}")
print(f"Maximum document length: {max(document_lengths):,}")
print(f"Average document length: {mean(document_lengths):,.2f}")

DOCUMENT INTEGRITY
Unique document IDs: 6
Duplicate document IDs: []
Documents with empty content: []

PAGE INTEGRITY
Total pages: 6
Duplicate pages: []
Missing document references: []
Empty pages: []

PAGE NUMBERING
5eaa2f3a4137bbd8: {'page_numbers': [1], 'is_sorted_contiguous': True}
4bc29707830b7f97: {'page_numbers': [1], 'is_sorted_contiguous': True}
6e0ae6c5f11c01b6: {'page_numbers': [1], 'is_sorted_contiguous': True}
1f8658d7b4d1d931: {'page_numbers': [1], 'is_sorted_contiguous': True}
17f4f629bc6c06aa: {'page_numbers': [1], 'is_sorted_contiguous': True}
b8c040cc92d5243c: {'page_numbers': [1], 'is_sorted_contiguous': True}

DOCUMENT-PAGE RELATIONSHIPS
Documents without pages: []
5eaa2f3a4137bbd8: 1 page(s)
4bc29707830b7f97: 1 page(s)
6e0ae6c5f11c01b6: 1 page(s)
1f8658d7b4d1d931: 1 page(s)
17f4f629bc6c06aa: 1 page(s)
b8c040cc92d5243c: 1 page(s)

CORPUS STATISTICS
Total document characters: 118,204
Total page characters: 118,204
Minimum page length: 9,497
Maximum page length: 32,36

### Result Interpretation — Input Validation and Corpus Audit

The corpus passed all integrity checks.

| Metric                      |    Result |
| --------------------------- | --------: |
| Documents                   |         6 |
| Pages                       |         6 |
| Total characters            |   118,204 |
| Minimum page length         |     9,497 |
| Maximum page length         |    32,360 |
| Average page length         | 19,700.67 |
| Duplicate document IDs      |         0 |
| Duplicate pages             |         0 |
| Missing document references |         0 |
| Empty documents or pages    |         0 |

### Structural Findings

* Each document currently contains exactly one logical page.
* All page numbers start at 1.
* Every page references an existing document.
* Document content and page content have matching total character counts.

### Technical Findings

1. The corpus is structurally valid and ready for chunking experiments.
2. Chunk provenance must preserve document IDs and page numbers.
3. Document length varies significantly, so chunking behavior must be evaluated across different input sizes.
4. Multi-page document handling should remain supported by the implementation, even though the current corpus does not exercise that case.

**Conclusion:** No ingestion or data-integrity issues were detected. The existing processed records can be used as the input for the chunking pipeline.

**Next Step:** Inspect Markdown structure to understand headings, paragraphs, lists, tables, and code blocks before selecting chunk boundaries.
